# Segmentación de núcleos celulares (MoNuSeg 2018): CNN vs U-Net, con y sin Data Augmentation

Proyecto del seminario de **Deep Learning para Bioimágenes (dlba-pucp)**.

**Objetivo:** comparar dos arquitecturas de segmentación semántica (una CNN encoder-decoder simple y una U-Net con skip connections) sobre el dataset público **MoNuSeg 2018** (imágenes de histopatología con máscaras de núcleos celulares), y medir el efecto de aplicar *Data Augmentation* (rotaciones aleatorias) sobre el desempeño de ambos modelos.

**Flujo del notebook:**
1. Configuración y descarga del dataset desde Kaggle
2. Emparejamiento de imágenes y máscaras
3. Dataset y DataLoader en PyTorch
4. Función de pérdida (Dice + BCE) y métrica IoU
5. Entrenamiento y evaluación: CNN simple vs U-Net (sin augmentation)
6. Data Augmentation (rotaciones aleatorias sincronizadas imagen-máscara)
7. Re-entrenamiento con augmentation y comparación final


## 1. Librerías y configuración general

In [ ]:
import os
import glob
import random
import xml.etree.ElementTree as ET

import numpy as np
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

IMG_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-3

## 2. Descarga del dataset desde Kaggle

Se usa el dataset **MoNuSeg 2018** (`tuanledinh/monuseg2018`), que contiene imágenes de histopatología (H&E) junto con sus máscaras de segmentación de núcleos celulares.

> ⚠️ **Necesitas tu propio `kaggle.json`** (credenciales de la API de Kaggle). Se obtiene desde tu cuenta de Kaggle en `Account → Settings → API → Create New Token`. El archivo se descarga automáticamente; súbelo cuando la celda lo solicite.

In [ ]:
import os
from google.colab import files

# 1. Borramos cualquier archivo corrupto o caché anterior para empezar desde cero
!rm -rf /content/dataset
!rm -rf ~/.cache/kagglehub
!rm -rf ~/.kaggle

# 2. Subimos el archivo de credenciales
print("Por favor, sube tu archivo kaggle.json:")
uploaded = files.upload()

# 3. Configuramos las credenciales correctamente
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Descargamos el dataset (usando el que aparece en tu HTML) y lo descomprimimos
!pip install -q kaggle
!kaggle datasets download -d tuanledinh/monuseg2018 --unzip -p /content/dataset

# 5. Verificamos que ahora sí sean las imágenes
print("\n--- Verificación del contenido ---")
!ls -lh /content/dataset

# Actualizamos la ruta para que la celda 3 pueda encontrar las imágenes
ruta = "/content/dataset"

## 3. Emparejamiento de imágenes y máscaras

El dataset descarga imágenes y máscaras en carpetas separadas; esta función las empareja por nombre de archivo, limpiando sufijos como `_mask` y espacios en blanco.

In [ ]:
import os
import glob

def encontrar_pares(carpeta):
    archivos = glob.glob(carpeta + "/**/*.*", recursive=True)

    # Separar imágenes y máscaras
    imagenes = [f for f in archivos if '/images/' in f.replace('\\', '/')]
    posibles_mascaras = [f for f in archivos if '/images/' not in f.replace('\\', '/')]

    # Construir diccionario de máscaras limpiando espacios traicioneros
    mascaras_dict = {}
    for m in posibles_mascaras:
        nombre = os.path.splitext(os.path.basename(m))[0]
        # .strip() elimina espacios al principio y al final
        nombre_limpio = nombre.replace('_mask', '').replace('_Mask', '').strip()
        mascaras_dict[nombre_limpio] = m

    # Emparejar
    pares = []
    for img in imagenes:
        nombre_img = os.path.splitext(os.path.basename(img))[0].strip()
        if nombre_img in mascaras_dict:
            pares.append((img, mascaras_dict[nombre_img]))

    return pares

pares = encontrar_pares(ruta)
print("Pares imagen-anotación encontrados:", len(pares))

## 4. Visualización de un ejemplo (imagen + máscara)

In [ ]:
img_path, mask_path = pares[0]

# Leer la imagen y la máscara usando OpenCV
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
mascara = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) # Leemos la máscara directo

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img); axes[0].set_title("Imagen")
axes[1].imshow(mascara, cmap="gray"); axes[1].set_title("Máscara")
plt.show()

## 5. Dataset y DataLoader de PyTorch

`MoNuSegDataset` lee cada par imagen-máscara, binariza la máscara, redimensiona a `IMG_SIZE x IMG_SIZE` y convierte todo a tensores normalizados en `[0, 1]`.

In [ ]:
class MoNuSegDataset(Dataset):
    def __init__(self, pares, tam=128):
        self.pares = pares
        self.tam = tam

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, idx):
        img_path, mask_path = self.pares[idx]

        # 1. Leer imágenes
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mascara = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Si la máscara viene de Kaggle con valores 0 a 255, la convertimos a 0 y 1 lógico
        if mascara.max() > 1:
            mascara = (mascara > 127).astype(np.uint8)

        # 2. Redimensionar
        img = cv2.resize(img, (self.tam, self.tam))
        mascara = cv2.resize(mascara, (self.tam, self.tam), interpolation=cv2.INTER_NEAREST)

        # 3. Convertir a tensores para PyTorch
        img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        mascara = torch.from_numpy(mascara).float().unsqueeze(0)

        return img, mascara

### 5.1 Partición train/test (80/20) y creación de los DataLoaders

In [ ]:
random.shuffle(pares)
n_test = int(0.2 * len(pares))
pares_test = pares[:n_test]
pares_train = pares[n_test:]

print(f"Total de pares encontrados: {len(pares)}")
print(f"Para entrenamiento: {len(pares_train)} | Para prueba: {len(pares_test)}")

train_ds = MoNuSegDataset(pares_train, tam=IMG_SIZE)
test_ds = MoNuSegDataset(pares_test, tam=IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

## 6. Función de pérdida: Dice + BCE

Se combina **Binary Cross-Entropy** (penaliza error píxel a píxel) con **Dice Loss** (penaliza el desbalance entre región de fondo y región de núcleos, que suele ser pequeña). Esta combinación es estándar en segmentación biomédica.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        # Aplanar los tensores
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calcular Dice
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)

        # Calcular BCE estándar
        BCE = nn.BCELoss()(inputs, targets)

        # Combinar ambas métricas
        return BCE + dice_loss

## 7. Funciones auxiliares: entrenamiento, evaluación (IoU) y visualización de predicciones

In [ ]:
def iou_score(pred, target, umbral=0.5):
    pred = (pred > umbral).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter / (union + 1e-6)).item()

def entrenar(modelo, train_loader, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.Adam(modelo.parameters(), lr=lr)
    criterio = DiceBCELoss()
    perdidas = []

    for epoch in range(epochs):
        modelo.train()
        perdida_epoca = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            salida = modelo(imgs)
            loss = criterio(salida, masks)
            loss.backward()
            optimizer.step()
            perdida_epoca += loss.item()

        perdida_epoca /= len(train_loader)
        perdidas.append(perdida_epoca)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {perdida_epoca:.4f}")

    return perdidas

def evaluar(modelo, test_loader):
    modelo.eval()
    ious = []
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            salida = modelo(imgs)
            ious.append(iou_score(salida, masks))
    return sum(ious) / len(ious)

def mostrar_predicciones(modelo, dataset, n=3, titulo="Modelo"):
    modelo.eval()

    # Protección: nunca pedir más imágenes de las que existen en el dataset
    n = min(n, len(dataset))
    if n == 0:
        print("El dataset está vacío. No hay predicciones para mostrar.")
        return

    fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))

    # Ajuste para cuando solo hay 1 imagen (evita errores de dimensionalidad en matplotlib)
    if n == 1:
        axes = np.array([axes])

    with torch.no_grad():
        for i in range(n):
            img, mask = dataset[i]
            salida = modelo(img.unsqueeze(0).to(DEVICE))
            pred = (salida > 0.5).float().cpu().squeeze().numpy()

            axes[i, 0].imshow(img.permute(1, 2, 0).numpy()); axes[i, 0].set_title("Imagen")
            axes[i, 1].imshow(mask.squeeze().numpy(), cmap="gray"); axes[i, 1].set_title("Ground truth")
            axes[i, 2].imshow(pred, cmap="gray"); axes[i, 2].set_title(titulo)
            for ax in axes[i]:
                ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Modelo 1 — CNN Encoder-Decoder simple

Arquitectura básica: 3 bloques convolucionales con `BatchNorm` + `MaxPool` como encoder, y 3 capas `ConvTranspose2d` como decoder. **No tiene skip connections**, por lo que pierde detalle espacial fino durante el downsampling.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Desactivamos el bias en estas capas porque el BatchNorm se encarga de esa constante
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout2d(0.2) # Apaga el 20% de las neuronas aleatoriamente para generalizar mejor

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.up2 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.up3 = nn.ConvTranspose2d(16, 8, 2, stride=2)
        self.salida = nn.Conv2d(8, 1, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Aplicamos la convolución, luego el BatchNorm, la activación y finalmente el MaxPool
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.dropout(x)
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        x = self.relu(self.up1(x))
        x = self.relu(self.up2(x))
        x = self.relu(self.up3(x))

        return torch.sigmoid(self.salida(x))

### 8.1 Entrenamiento y evaluación de la CNN (sin Data Augmentation)

In [ ]:
modelo_cnn = CNN().to(DEVICE)
perdidas_cnn = entrenar(modelo_cnn, train_loader)
iou_cnn = evaluar(modelo_cnn, test_loader)
print(f"CNN simple - IoU en test: {iou_cnn:.4f}")

mostrar_predicciones(modelo_cnn, test_ds, n=3, titulo="CNN simple")

## 9. Modelo 2 — U-Net

Misma profundidad que la CNN, pero con **skip connections** (`torch.cat`) que concatenan los mapas de características del encoder con los del decoder, preservando bordes y detalles finos de los núcleos.

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 16, 3, padding=1)
        self.enc2 = nn.Conv2d(16, 32, 3, padding=1)
        self.enc3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.Conv2d(64, 32, 3, padding=1)   # 32 (up) + 32 (skip)

        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = nn.Conv2d(32, 16, 3, padding=1)   # 16 (up) + 16 (skip)

        self.salida = nn.Conv2d(16, 1, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        e1 = self.relu(self.enc1(x))       # 128x128
        p1 = self.pool(e1)                 # 64x64
        e2 = self.relu(self.enc2(p1))      # 64x64
        p2 = self.pool(e2)                 # 32x32
        e3 = self.relu(self.enc3(p2))      # 32x32 (cuello de botella)

        d2 = self.up2(e3)                  # 64x64
        d2 = torch.cat([d2, e2], dim=1)    # skip connection
        d2 = self.relu(self.dec2(d2))

        d1 = self.up1(d2)                  # 128x128
        d1 = torch.cat([d1, e1], dim=1)    # skip connection
        d1 = self.relu(self.dec1(d1))

        return torch.sigmoid(self.salida(d1))

### 9.1 Entrenamiento y evaluación de la U-Net (sin Data Augmentation)

In [ ]:
modelo_unet = UNet().to(DEVICE)
perdidas_unet = entrenar(modelo_unet, train_loader)
iou_unet = evaluar(modelo_unet, test_loader)
print(f"U-Net simple - IoU en test: {iou_unet:.4f}")

mostrar_predicciones(modelo_unet, test_ds, n=3, titulo="U-Net simple")

## 10. Comparación inicial: CNN vs U-Net (sin Data Augmentation)

In [ ]:
print(f"CNN simple - IoU: {iou_cnn:.4f}")
print(f"U-Net simple - IoU: {iou_unet:.4f}")

plt.plot(perdidas_cnn, label="CNN simple")
plt.plot(perdidas_unet, label="U-Net simple")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.title("Comparación de entrenamiento")
plt.show()

In [ ]:
# Guardamos los resultados "sin augmentation" antes de reentrenar
# (las variables iou_cnn / iou_unet se sobrescribirán en la siguiente sección)
iou_cnn_base = iou_cnn
iou_unet_base = iou_unet


## 11. Data Augmentation

Se redefine `MoNuSegDataset` agregando un parámetro `augment`. Cuando está activo, se aplica una **rotación aleatoria sincronizada** (mismo ángulo, entre -90° y 90°) a la imagen y a su máscara con `torchvision.transforms.functional.rotate`, aumentando la variabilidad del set de entrenamiento sin alterar la correspondencia píxel a píxel entre imagen y máscara.

In [ ]:
import torchvision.transforms.functional as TF
import random
import torch
import cv2
import numpy as np
from torch.utils.data import Dataset

class MoNuSegDataset(Dataset):
    def __init__(self, pares, tam=128, augment=False):
        self.pares = pares
        self.tam = tam
        self.augment = augment  # Activa o desactiva el Data Augmentation

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, idx):
        img_path, mask_path = self.pares[idx]

        # 1. Leer imágenes
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mascara = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Binarizar la máscara
        if mascara.max() > 1:
            mascara = (mascara > 127).astype(np.uint8)

        # 2. Redimensionar
        img = cv2.resize(img, (self.tam, self.tam))
        mascara = cv2.resize(mascara, (self.tam, self.tam), interpolation=cv2.INTER_NEAREST)

        # 3. Convertir a tensores para PyTorch
        img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        mascara = torch.from_numpy(mascara).float().unsqueeze(0)

        # 4. Data Augmentation (Rotaciones Aleatorias)
        if self.augment:
            # 50% de probabilidad de aplicar una rotación en esta iteración
            if random.random() > 0.5:
                # Elegir un ángulo aleatorio entre -90 y 90 grados
                angulo = random.uniform(-90, 90)

                # Aplicar exactamente la misma rotación a ambas partes
                img = TF.rotate(img, angulo)
                mascara = TF.rotate(mascara, angulo)

        return img, mascara

## 12. Re-creación de los DataLoaders con Data Augmentation

El augmentation se activa **solo en train** (`augment=True`); el set de test se mantiene sin augmentation para evaluar de forma justa sobre datos "limpios".

In [ ]:
train_ds = MoNuSegDataset(pares_train, tam=IMG_SIZE, augment=True) # Data Augmentation activado
test_ds = MoNuSegDataset(pares_test, tam=IMG_SIZE, augment=False)  # Siempre en False para Test

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

### 12.1 CNN — reentrenamiento con Data Augmentation

In [ ]:
modelo_cnn = CNN().to(DEVICE)
perdidas_cnn = entrenar(modelo_cnn, train_loader)
iou_cnn = evaluar(modelo_cnn, test_loader)
print(f"CNN simple - IoU en test: {iou_cnn:.4f}")

mostrar_predicciones(modelo_cnn, test_ds, n=3, titulo="CNN simple")

### 12.2 U-Net — reentrenamiento con Data Augmentation

In [ ]:
modelo_unet = UNet().to(DEVICE)
perdidas_unet = entrenar(modelo_unet, train_loader)
iou_unet = evaluar(modelo_unet, test_loader)
print(f"U-Net simple - IoU en test: {iou_unet:.4f}")

mostrar_predicciones(modelo_unet, test_ds, n=3, titulo="U-Net simple")

## 13. Comparación final: CNN vs U-Net (con Data Augmentation)

In [ ]:
print(f"CNN simple - IoU: {iou_cnn:.4f}")
print(f"U-Net simple - IoU: {iou_unet:.4f}")

plt.plot(perdidas_cnn, label="CNN simple")
plt.plot(perdidas_unet, label="U-Net simple")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.title("Comparación de entrenamiento")
plt.show()

## 14. Conclusión — Resumen global (4 configuraciones)

Se compara el IoU de ambos modelos, con y sin Data Augmentation, en un solo gráfico de barras para visualizar el efecto combinado de la arquitectura (skip connections) y del augmentation sobre el desempeño de segmentación.

In [ ]:
resultados = {
    "CNN\n(sin aug.)": iou_cnn_base,
    "U-Net\n(sin aug.)": iou_unet_base,
    "CNN\n(con aug.)": iou_cnn,
    "U-Net\n(con aug.)": iou_unet,
}

nombres = list(resultados.keys())
valores = list(resultados.values())
colores = ["#8fbcd4", "#4a7d96", "#f4a582", "#b2182b"]

plt.figure(figsize=(7, 5))
barras = plt.bar(nombres, valores, color=colores)
for barra, v in zip(barras, valores):
    plt.text(barra.get_x() + barra.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")

plt.ylabel("IoU en test")
plt.title("Comparación global: arquitectura x Data Augmentation")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print("Resumen de resultados:")
for nombre, v in resultados.items():
    print(f"  {nombre.replace(chr(10), ' ')}: IoU = {v:.4f}")
